# Community Detection

## Learning Objectives

1. **Define** modularity Q and explain why maximizing it finds communities
2. **Implement** modularity computation for a given partition
3. **Describe** the Louvain algorithm conceptually (implementation in algorithm notebook)
4. **Compare** modularity with conductance for community quality

## What is a Community?

A **community** is a set of nodes more densely connected internally than to the rest of the graph.

There is no single formal definition — communities are task-dependent:
- Social: groups of friends
- Web: topic-related pages
- Biology: functional protein modules

**Challenge:** the right number of communities is not known in advance.

In [ ]:
from collections import defaultdict

def modularity(adj, partition):
    """
    Compute Newman-Girvan modularity Q.
    adj: dict node -> set of neighbors
    partition: dict node -> community_id
    Returns Q in [-1, 1]
    """
    m = sum(len(nbrs) for nbrs in adj.values()) / 2  # total edges
    if m == 0: return 0.0
    degree = {u: len(adj[u]) for u in adj}
    communities = defaultdict(set)
    for node, comm in partition.items():
        communities[comm].add(node)
    Q = 0.0
    for comm_nodes in communities.values():
        nodes = list(comm_nodes)
        for i, u in enumerate(nodes):
            for v in nodes:
                A_uv = 1 if v in adj[u] else 0
                Q += A_uv - (degree[u] * degree[v]) / (2 * m)
    return Q / (2 * m)

# Build two-community graph
adj = defaultdict(set)
# Community 1: nodes 0-4, dense
c1 = [0,1,2,3,4]
for i in c1:
    for j in c1:
        if i<j:
            adj[i].add(j); adj[j].add(i)
# Community 2: nodes 5-9, dense
c2 = [5,6,7,8,9]
for i in c2:
    for j in c2:
        if i<j:
            adj[i].add(j); adj[j].add(i)
# Few inter-community edges
for u,v in [(2,5),(3,7)]:
    adj[u].add(v); adj[v].add(u)

# True partition
true_part = {n:0 for n in c1}
true_part.update({n:1 for n in c2})
# Random partition
rand_part = {n: n%3 for n in range(10)}
# All-one partition
all_one   = {n: 0 for n in range(10)}

print(f"True partition Q:   {modularity(adj, true_part):.4f}")
print(f"Random partition Q: {modularity(adj, rand_part):.4f}")
print(f"All-one Q:          {modularity(adj, all_one):.4f}")
print("(Higher Q = better community structure)")

## Modularity Q

$$Q = \frac{1}{2m} \sum_{u,v} \left[ A_{uv} - \frac{d_u d_v}{2m} \right] \delta(c_u, c_v)$$

- $A_{uv}$ = 1 if edge $(u,v)$ exists
- $d_u$ = degree of node $u$
- $m$ = total number of edges
- $\delta(c_u, c_v)$ = 1 if $u$ and $v$ are in the same community

**Interpretation:** Q > 0 means more edges within communities than expected by chance.
Q ∈ [−1, 1]; values > 0.3 typically indicate meaningful structure.

**Null model:** $d_u d_v / 2m$ = expected number of edges between $u$ and $v$ in a random graph with the same degree sequence.

## Louvain Algorithm (Summary)

The full implementation is in `ml_010_02_louvain_algorithm.ipynb`. Here is the conceptual overview:

**Phase 1 — Local greedy optimization:**
For each node $u$, try moving it to each neighbor's community.
Keep the move if it increases Q.
Repeat until no improvement.

**Phase 2 — Community aggregation:**
Collapse each community into a single super-node.
Weights = number of edges between communities.
Return to Phase 1 on the reduced graph.

**Repeat** until Q no longer improves.

**Complexity:** nearly O(n log n) in practice.
**Weakness:** resolution limit — cannot find communities smaller than √(2m) nodes.

## Conductance

An alternative to modularity that avoids the resolution limit:

$$\phi(S) = \frac{\text{cut}(S, \bar{S})}{\min(\text{vol}(S), \text{vol}(\bar{S}))}$$

where $\text{cut}(S,\bar{S})$ = edges crossing the boundary and $\text{vol}(S) = \sum_{u\in S} d_u$.

- $\phi = 0$: perfectly isolated community
- $\phi = 1$: all edges cross the boundary

**Network Community Profile (NCP):** plot minimum conductance vs community size to characterize multi-scale structure.

In [ ]:
def conductance(adj, S):
    S = set(S)
    Sbar = set(adj.keys()) - S
    cut = sum(1 for u in S for v in adj[u] if v in Sbar)
    vol_S    = sum(len(adj[u]) for u in S)
    vol_Sbar = sum(len(adj[u]) for u in Sbar)
    denom = min(vol_S, vol_Sbar)
    return cut / denom if denom > 0 else 0.0

# Compare true vs random cuts
print(f"Conductance of true community (nodes 0-4):    {conductance(adj, c1):.4f}")
print(f"Conductance of random cut (nodes 0,2,4,6,8): {conductance(adj, [0,2,4,6,8]):.4f}")
print()
# Sweep all sizes to show NCP pattern
print("Community size vs conductance (exhaustive, small graph):")
from itertools import combinations
best_by_size = {}
all_nodes = list(adj.keys())
for size in range(2, len(all_nodes)-1):
    for subset in combinations(all_nodes, size):
        c = conductance(adj, subset)
        if size not in best_by_size or c < best_by_size[size]:
            best_by_size[size] = c
for size, c in sorted(best_by_size.items()):
    bar = '#'*int(c*20)
    print(f"  size={size}: {c:.3f} {bar}")

## Summary

| Method | Objective | Strength | Weakness |
|--------|-----------|---------|---------|
| Modularity (Louvain) | Maximize Q | Fast; hierarchical | Resolution limit |
| Conductance (spectral) | Minimize φ | Scale-invariant | Slower; one community at a time |
| Girvan-Newman | Remove high-betweenness edges | Interpretable | O(mn) — very slow |

For large graphs, **Louvain** (modularity) is the practical default.
**Conductance** is preferred for evaluating community quality and understanding multi-scale structure.